# Reproduce paper numbers

This short notebook regenerates every number cited in *Who Turns to AI for Schoolwork?* directly from the shipped `data/processed/merged_dataset.csv`. It does **not** require any of the upstream API credentials. End-to-end runtime: under one minute on a laptop.

Each section is labeled with the paper section / table / figure it reproduces.

If you need to regenerate the data pipeline from upstream sources, use `01_full_pipeline.ipynb` instead.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)

# Resolve the data path whether the notebook is run from the repo root or from notebooks/
for candidate in [Path("data/processed/merged_dataset.csv"),
                  Path("../data/processed/merged_dataset.csv")]:
    if candidate.exists():
        DATA_PATH = candidate
        break
else:
    raise FileNotFoundError("Could not locate merged_dataset.csv")

df = pd.read_csv(DATA_PATH, index_col=0)
print(f"Loaded {len(df)} DMAs from {DATA_PATH}")
print(f"NaN search_interest count: {df['search_interest'].isna().sum()}  (treated as zero in the headline supervised model)")

## Section 5.3 / Table 1 — Five-cluster typology

In [ ]:
summary = df.groupby("cluster_id").agg(
    n=("dma_name", "size"),
    median_income=("median_income", "mean"),
    bach_plus_rate=("bach_plus_rate", "mean"),
    pct_nh_asian=("pct_nh_asian", "mean"),
    population=("population", "mean"),
    mean_interest=("search_interest", "mean"),
)
summary

## Appendix B / Table tab:clusterverify — Cluster geography

State distributions, Southern-state share, and racial-composition means by cluster. These verify the descriptive labels used in Section 5.3 (e.g. Cluster 4 as a Black-Belt-region non-metropolitan cluster, Cluster 0 as rural and small-metropolitan).

In [ ]:
SOUTH = {"AL", "AR", "FL", "GA", "KY", "LA", "MS", "NC",
         "SC", "TN", "VA", "WV", "TX", "OK"}
RACE_COLS = ["pct_hispanic", "pct_nh_white", "pct_nh_black", "pct_nh_asian"]

def state_of(name):
    parts = str(name).strip().split()
    return parts[-1] if parts and len(parts[-1]) == 2 and parts[-1].isupper() else ""

df["state"] = df["dma_name"].apply(state_of)

for cid in sorted(df["cluster_id"].dropna().unique()):
    sub = df[df["cluster_id"] == cid]
    states = sub["state"].value_counts()
    south_share = sub["state"].isin(SOUTH).mean()
    print(f"\nCluster {int(cid)} (n={len(sub)}):")
    print(f"  Top states: {dict(states.head(7))}")
    print(f"  Southern share: {south_share:.0%}")
    print("  Racial means: " + ", ".join(
        f"{c}={sub[c].mean():.1f}" for c in RACE_COLS))

## Section 5.3 — Cluster 0 rurality threshold

Supports the "rural and small-metropolitan" label by showing that 74% of Cluster 0 DMAs have populations below 1 million.

In [ ]:
print(f"National DMA median pop: {df['population'].median():,.0f}")
print(f"National DMA mean pop:   {df['population'].mean():,.0f}\n")

for cid in sorted(df["cluster_id"].dropna().unique()):
    sub = df[df["cluster_id"] == cid]
    print(f"Cluster {int(cid)} (n={len(sub)}):")
    print(f"  median pop = {sub['population'].median():,.0f}")
    print(f"  mean pop   = {sub['population'].mean():,.0f}")
    for thresh, label in [(500_000, '<500K'), (1_000_000, '<1M'), (2_500_000, '<2.5M')]:
        share = (sub['population'] < thresh).mean()
        print(f"  {label}: {share:.0%}")
    print()

## Appendix C / Table tab:vif — Variance inflation factors

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

SUPERVISED = ["median_income", "bach_plus_rate", "population", "pct_hispanic",
              "pct_nh_black", "pct_nh_asian", "score_all_ela"]

X = df[SUPERVISED].copy()
X = X.fillna(X.mean())
Xz = (X - X.mean()) / X.std()

vifs = pd.DataFrame({
    "feature": SUPERVISED,
    "VIF": [variance_inflation_factor(Xz.values, i) for i in range(len(SUPERVISED))],
}).sort_values("VIF", ascending=False)
vifs.round(2)

## Figure 1 — LOWESS panel

Regenerates the four-panel LOWESS figure (median household income first, followed by bachelor-plus rate, graduate/professional rate, and Asian share on log scale). The PDF saved to `paper/figures/fig_lowess.pdf` is built by the standalone script `paper/build_lowess_figure.py`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

PANELS = [
    ("median_income", "Median household income (USD)", False),
    ("bach_plus_rate", "Adults with bachelor's degree or higher (%)", False),
    ("grad_prof_rate", "Adults with graduate or professional degree (%)", False),
    ("pct_nh_asian", "Non-Hispanic Asian population share (%, log scale)", True),
]

predictors = [p for p, _, _ in PANELS]
imp = SimpleImputer(strategy="mean")
X = pd.DataFrame(imp.fit_transform(df[predictors]),
                 columns=predictors, index=df.index)
y = df["search_interest"].fillna(0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, (col, xlabel, log_x) in zip(axes, PANELS):
    x = X[col]
    x_plot = np.log10(x.replace(0, np.nan)) if log_x else x
    sns.regplot(
        x=x_plot, y=y, ax=ax, lowess=True,
        scatter_kws={"alpha": 0.45, "s": 22, "color": "#3a6ea5"},
        line_kws={"color": "#c0392b", "linewidth": 2.2},
    )
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel("")
    ax.tick_params(axis="both", labelsize=9)
    ax.grid(True, alpha=0.25, linestyle="--")
    if col == "median_income":
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${int(v/1000)}k"))
axes[0].set_ylabel("AI-for-schoolwork search interest", fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()

## Headline supervised model — Table 2 / Section 5.4

Five regressors compared under 5-fold CV with `random_state=42`. Tuned gradient boosting is the headline.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

TARGET = "search_interest"
y = df[TARGET].fillna(0).values
X = df[SUPERVISED].fillna(df[SUPERVISED].mean()).values

outer = KFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Linear":         LinearRegression(),
    "Ridge":          Ridge(),
    "Lasso":          Lasso(),
    "Random Forest":  RandomForestRegressor(random_state=42),
    "GBM (default)":  GradientBoostingRegressor(random_state=42),
}
rows = []
for name, model in models.items():
    r2 = cross_val_score(model, X, y, cv=outer, scoring="r2", n_jobs=-1)
    mae = -cross_val_score(model, X, y, cv=outer, scoring="neg_mean_absolute_error", n_jobs=-1)
    rows.append({"Model": name, "R2_mean": r2.mean(), "R2_sd": r2.std(), "MAE": mae.mean()})

# Tuned GBM via inner-CV grid search
PARAM_GRID = {
    "n_estimators":     [100, 200, 300],
    "learning_rate":    [0.01, 0.1, 0.2],
    "max_depth":        [3, 5, 7],
    "min_samples_split": [2, 5],
}
inner = KFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(GradientBoostingRegressor(random_state=42),
                    PARAM_GRID, cv=inner, scoring="r2", n_jobs=-1)
grid.fit(X, y)
best = grid.best_estimator_
r2 = cross_val_score(best, X, y, cv=outer, scoring="r2", n_jobs=-1)
mae = -cross_val_score(best, X, y, cv=outer, scoring="neg_mean_absolute_error", n_jobs=-1)
rows.append({"Model": "GBM (tuned)", "R2_mean": r2.mean(), "R2_sd": r2.std(), "MAE": mae.mean()})

pd.DataFrame(rows).round(3)

Headline: tuned GBM achieves $R^2 \approx 0.42 \pm 0.12$, comfortably above any linear baseline at $R^2 \approx 0.16$. See paper §5.4.